# 04 Relevance Review and Final Enrichment Consolidation

## Where is my data now?

The dataset has already gone through:

1. API collection from Europeana (`00_data_collection.ipynb`)
2. Deduplication by title (`01_data_access.ipynb`)
3. Cleaning and derived-variable creation (`02_data_cleaning_and_analysis.ipynb`)
4. Initial top-15-provider Wikidata enrichment (`03_wikidata_enhancement.ipynb`)

The current input for this notebook is:

`data/processed/europeana_india_unique_titles_enhanced_top15_providers.csv`

(245 rows, 16 columns; match-status counts: matched 139, uncertain 28, not_checked 78)

## What problem does this notebook address?

Notebook `02_data_cleaning_and_analysis.ipynb` already flagged a known ambiguity: the
word "Indian" in English can refer either to India or to Indigenous peoples of the
Americas, and the `possible_false_positive` column marks 17 records (out of 245) where
this ambiguity might apply.

That flag was created but never resolved. This notebook does three things:

1. **Continues provider enrichment** for a further batch of `not_checked` and
   `uncertain` `dataProvider` values, following the same Wikidata-verification
   approach as `03_wikidata_enhancement.ipynb`.
2. **Resolves the `possible_false_positive` flag** by manually reviewing all 17
   records and recording a reasoned relevance judgement for each one, rather than
   leaving the flag unresolved or silently dropping the records.
3. **Adds metadata-completeness indicators** that make gaps in the underlying
   Europeana metadata (e.g. the `subject` field) explicit and queryable.

## Rules followed in this notebook

Consistent with the rest of the project:

- The original 16 columns and all 245 rows are never overwritten or deleted.
- Wikidata matches are only recorded where there is verifiable evidence; unclear
  cases are marked `uncertain` or `needs_manual_review` rather than guessed.
- Every enrichment decision is documented with a short reasoning note.
- Outputs are saved to new, clearly named files - the original processed file is
  left untouched.


In [8]:
# Import required libraries
import pandas as pd
from pathlib import Path

print("Libraries imported successfully.")


Libraries imported successfully.


In [9]:
current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_dir = current_folder.parent
else:
    project_dir = current_folder

processed_dir = project_dir / "data" / "processed"

input_file = processed_dir / "europeana_india_unique_titles_enhanced_top15_providers.csv"

df = pd.read_csv(input_file)

print("Current working folder:", current_folder)
print("Input file:", input_file)
print("Input rows:", len(df))
print("Input columns:", len(df.columns))
print("match_status counts:")
print(df["match_status"].value_counts(dropna=False))


Current working folder: C:\Users\kevin\Documents\GitHub\Open_Project_2026\notebooks
Input file: C:\Users\kevin\Documents\GitHub\Open_Project_2026\data\processed\europeana_india_unique_titles_enhanced_top15_providers.csv
Input rows: 245
Input columns: 16
match_status counts:
match_status
matched        139
not_checked     78
uncertain       28
Name: count, dtype: int64


## Part 1 - Continued provider enrichment

The previous notebook enriched providers in order of frequency but left 78 records
`not_checked` and 28 `uncertain`. Here, a further batch of the most frequent
`not_checked` providers (10 providers, 24 records) and all 4 `uncertain` providers
(28 records) were manually researched against Wikidata.

The full reasoning for every provider is recorded in `provider_enrichment_notes`.
Two decisions are worth calling out specifically, since they reflect the project's
"preserve uncertainty" principle rather than maximising the match count:

- **Map Collection UK** (3 records) could not be matched to any specific institution
  via Wikidata - it reads as a generic label rather than a formal name - so it was
  moved to `needs_manual_review` instead of being force-matched.
- **National Audiovisual Institute France** (2 records) was initially matched to
  Institut national de l'audiovisuel (INA, Q1665109), but on review the exact English
  provider string could not be verified verbatim on Wikidata, so it was deliberately
  *downgraded* back to `uncertain` rather than kept as a confident match.


In [10]:
df_enriched = df.copy()

if "provider_enrichment_notes" not in df_enriched.columns:
    df_enriched["provider_enrichment_notes"] = pd.NA

# --- Batch: previously not_checked providers, now resolved against Wikidata ---
not_checked_updates = {
    "Marciana National Library": (
        "Q578460", "Biblioteca Nazionale Marciana", "national library", "matched",
        "Verified via Wikidata; state library of Italy in Venice; confirmed via Europeana's own organisation page."
    ),
    "Bavarian State Library": (
        "Q256507", "Bayerische Staatsbibliothek (Bavarian State Library)", "national library", "matched",
        "Classified as 'national library' for consistency with existing dataset category usage. "
        "Note for transparency: technically a Land (state) library of Bavaria, not Germany's national "
        "library (that is the German National Library, DNB)."
    ),
    "Bauhaus University Weimar. University Library": (
        "Q1581267", "Weimar University Library (Universitaetsbibliothek Weimar)", "university library", "matched",
        "Exact string match confirmed on Europeana's own organisation page for this provider."
    ),
    "National Audiovisual Institute France": (
        "Q1665109", "Institut national de l'audiovisuel (INA) [candidate, unconfirmed]",
        "film archive / media institution", "uncertain",
        "Downgraded to uncertain on review: strong candidate is INA (Q1665109) but the exact English "
        "provider string was not found verbatim on Wikidata, so kept as a suggested candidate rather "
        "than a confirmed match."
    ),
    "DFF – German Film Institute & Film Museum": (
        "Q1205634", "DFF – Deutsches Filminstitut & Filmmuseum", "film archive / media institution", "matched",
        "Exact institutional name match (English translation)."
    ),
    "Tartu Art Museum": (
        "Q12376420", "Tartu Kunstimuuseum (Tartu Art Museum)", "museum", "matched",
        "Distinct state art museum in Tartu, Estonia (est. 1940), separate from Art Museum of Estonia in Tallinn."
    ),
    "Art Museum of Estonia": (
        "Q1754105", "Eesti Kunstimuuseum (Art Museum of Estonia)", "museum organisation", "matched",
        "National umbrella institution in Tallinn overseeing 5 branch museums (Kumu, Kadriorg, etc.)."
    ),
    "National Library of France": (
        "Q193563", "Bibliotheque nationale de France (BnF)", "national library", "matched",
        "Direct, unambiguous translation of BnF."
    ),
    "Wellcome Collection": (
        "Q7981191", "Wellcome Collection", "museum", "matched",
        "London museum and library (part of Wellcome Trust); classified as museum as primary identity."
    ),
    "Map Collection UK": (
        pd.NA, pd.NA, pd.NA, "needs_manual_review",
        "No specific institution identified via research; label appears generic/non-standard rather than "
        "a formal institution name. Left unmatched deliberately - do not guess a QID."
    ),
}

rows_changed_1 = 0
for provider, (qid, label, itype, status, note) in not_checked_updates.items():
    mask = (df_enriched["dataProvider"] == provider) & (df_enriched["match_status"] == "not_checked")
    n = mask.sum()
    if pd.notna(qid):
        df_enriched.loc[mask, "wikidata_qid"] = qid
        df_enriched.loc[mask, "wikidata_label"] = label
        df_enriched.loc[mask, "institution_type"] = itype
    elif itype is not None and pd.notna(itype):
        df_enriched.loc[mask, "institution_type"] = itype
    df_enriched.loc[mask, "match_status"] = status
    df_enriched.loc[mask, "provider_enrichment_notes"] = note
    rows_changed_1 += n

print("Rows updated in not_checked review:", rows_changed_1)


Rows updated in not_checked review: 24


In [11]:
# --- Batch: previously uncertain providers, now resolved against Wikidata ---
uncertain_updates = {
    "German Documentation Center for Art History - Marburg Picture Index": (
        "Q28734805",
        "Deutsches Dokumentationszentrum fuer Kunstgeschichte - Bildarchiv Foto Marburg",
        None, "matched",
        "Exact match confirmed on Europeana's own organisation page (identical provider name). "
        "Note: Wikidata also has a closely related second item (Q860570) for what appears to be the "
        "same institution from a different angle - flagged as a Wikidata data-quality issue, not hidden."
    ),
    "Cinecittà - Luce": (
        "Q781183", "Istituto Luce Cinecitta (Historical Archive Luce)", None, "matched",
        "Wikidata Q781183 explicitly describes the merged 'Istituto Luce Cinecitta (Italy) Historical "
        "Archive Luce' entity (Istituto Luce and Cinecitta merged 2009-2011). Moderate-high confidence."
    ),
    "TopFoto": (
        "Q110975090", "TopFoto (trading name of Topham Partners LLP)", "photographic archive", "matched",
        "High confidence on identity (UK photo agency founded 1927). Institution type set to "
        "'photographic archive'. IMPORTANT INTERPRETIVE NOTE: TopFoto is a private, commercial photo "
        "agency/company, not a public museum/library/archive-sector heritage institution."
    ),
    "Toy Museum of the City of Nuremberg": (
        "Q2310413", "Nuernberger Spielzeugmuseum (Nuremberg Toy Museum)", None, "matched",
        "Wikidata explicitly lists 'Spielzeugmuseum der Stadt Nuernberg' as an alternate name - exact "
        "match, high confidence."
    ),
}

rows_changed_2 = 0
for provider, (qid, label, itype, status, note) in uncertain_updates.items():
    mask = (df_enriched["dataProvider"] == provider) & (df_enriched["match_status"] == "uncertain")
    n = mask.sum()
    df_enriched.loc[mask, "wikidata_qid"] = qid
    df_enriched.loc[mask, "wikidata_label"] = label
    if itype is not None:
        df_enriched.loc[mask, "institution_type"] = itype
    df_enriched.loc[mask, "match_status"] = status
    df_enriched.loc[mask, "provider_enrichment_notes"] = note
    rows_changed_2 += n

print("Rows updated in uncertain review:", rows_changed_2)
print()
print("match_status counts after Part 1:")
print(df_enriched["match_status"].value_counts(dropna=False))

assert len(df_enriched) == len(df), "Row count changed unexpectedly."


Rows updated in uncertain review: 28

match_status counts after Part 1:
match_status
matched                186
not_checked             54
needs_manual_review      3
uncertain                2
Name: count, dtype: int64


## Part 2 - Resolving `possible_false_positive`: the "Indianer" finding

All 17 records flagged by `possible_false_positive` were manually reviewed. Every one
turned out to be a genuine false positive with respect to India - but not for random
reasons. All 17 follow a specific, systematic pattern: the search terms "Indian
textile" and "Indian art" also match records where European (mostly German)
cataloguing institutions use "Indianer" (German) - a term used in German-language
ethnographic cataloguing for **Indigenous peoples of the Americas** - not people or
objects connected to India.

The evidence is consistent and unambiguous in every case: specific place names
(Panama/Kuna Yala, Brazil/Camacan, Guatemala/Kekchi, the Gran Chaco region,
Mexico/Huichol, Apache in the US Southwest) or explicit context (e.g. "Amerika" in a
description field, a "Cowboy/Indianer" pairing).

**Finding:** 17 of 245 records (approximately 6.9% of this deduplicated dataset) that
were captured by an India-related search query are not related to India. This is not
random noise - it is a specific, citable artefact of how an English search term
("Indian") intersects with an unrelated German cataloguing convention
("Indianer" = Indigenous peoples of the Americas). This finding is discussed further
in the Limitations section at the end of this notebook.

Per the project's rules, these 17 records are **retained**, not deleted. They are
flagged via a new `record_relevance_status` column so that later analysis can exclude
them from India-specific figures while keeping them visible and auditable in the
saved dataset.


In [12]:
if "record_relevance_status" not in df_enriched.columns:
    df_enriched["record_relevance_status"] = pd.NA
if "relevance_notes" not in df_enriched.columns:
    df_enriched["relevance_notes"] = pd.NA

# Manually reviewed relevance notes, keyed by (title, search_term) rather than row
# index, so this remains correct even if row order changes on re-run.
relevance_reviews = [
    ("Indianer-Kunst", "Indian art",
     "German 'Indianer' (Indigenous American) rather than 'Inder' (person from India); "
     "no place context given but term is unambiguous in German usage."),
    ("tryckteknik, Camacan-indianer, indian, konst, teckning, Teckning, drawing", "Indian art",
     "Explicitly Brazilian (Camacan people); source citation is a 19th-century Brazil "
     "travelogue (Prinz zu Wied-Neuwied)."),
    ("Indianer", "Indian art",
     "Bare German term 'Indianer', no further context; consistent with this provider's "
     "broader usage pattern for Indigenous-American imagery."),
    ("Cowboy/Indianer", "Indian art",
     "Explicit 'Cowboy/Indianer' pairing signals North American Old West imagery, not India."),
    ("Kekchi Indianer: Tanzmaske", "Indian art",
     "Kekchi (Q'eqchi') is a Maya people of Guatemala."),
    ("Kuna Yala - The textile molas show Indian motives", "Indian textile",
     "Explicitly Kuna people of Panama, described in the source metadata as 'indianische "
     "Ureinwohner' (Indigenous inhabitants)."),
    ("weapons, crafts, textile work, huicholic Indian, ornamental objects, photography,", "Indian textile",
     "Huichol people of Mexico; description field explicitly says 'Amerika'."),
    ("Kuna-Indianerfrau in Panama", "Indian textile",
     "Explicit 'Panama' in title; Kuna Indigenous community."),
    ("An Apache Indian, Go-Shona, in ceremonial dress.", "Indian textile",
     "Apache is a well-known Native American nation of the U.S. Southwest."),
    ("Kuna Yala - Textile Molas zeigen Tiere oder indianische Motive", "Indian textile",
     "Part of the same 2011 Kuna Yala (Panama) field-trip photo series."),
    ("Kuna Yala - Kuna-Indianerfrau mit Molas", "Indian textile",
     "Same Kuna Yala (Panama) photo series."),
    ("Kuna Yala - Molas mit indianischen Motiven", "Indian textile",
     "Same Kuna Yala (Panama) photo series."),
    ("Kuna Yala - Kuna-Indianerin mit Molas", "Indian textile",
     "Same Kuna Yala (Panama) photo series."),
    ("Kuna Yala - Molas with Indian motifs", "Indian textile",
     "Same Kuna Yala (Panama) photo series (English-language variant)."),
    ("Indianerleben im Gran Chaco", "Indian textile",
     "Gran Chaco is a South American region spanning Argentina, Paraguay, and Bolivia."),
    ("Kuna Yala - Kuna-Indian and German tourist", "Indian textile",
     "Same Kuna Yala (Panama) photo series."),
    ("Kuna Yala - Porträt einer alten Kuna-Indianerin", "Indian textile",
     "Same Kuna Yala (Panama) photo series."),
]

matched_count = 0
for title, search_term, note in relevance_reviews:
    mask = (df_enriched["title"] == title) & (df_enriched["search_term"] == search_term)
    n = mask.sum()
    if n == 0:
        print(f"WARNING: no row matched for title={title!r}")
        continue
    df_enriched.loc[mask, "record_relevance_status"] = "likely_not_india_related"
    df_enriched.loc[mask, "relevance_notes"] = note
    matched_count += n

print("Records reviewed and flagged:", matched_count)
print()
print("record_relevance_status counts:")
print(df_enriched["record_relevance_status"].value_counts(dropna=False))

assert matched_count == df_enriched["possible_false_positive"].sum() == 17, \
    "Every possible_false_positive record should have been matched and reviewed."
assert len(df_enriched) == len(df), "Row count changed unexpectedly."


Records reviewed and flagged: 17

record_relevance_status counts:
record_relevance_status
<NA>                        228
likely_not_india_related     17
Name: count, dtype: int64


## Part 3 - Metadata-completeness indicators

Two lightweight, deterministic completeness columns are added here to make gaps in
the source metadata explicit rather than implicit:

- `has_subject` - whether the `subject` field is populated. As shown below, this is
  `False` for every one of the 245 records: `subject` contributes no usable signal
  in this dataset at all, despite being one of Europeana's core descriptive
  properties. This connects directly to the false-positive finding above and is
  discussed further in the Limitations section.
- `has_country`, `description_word_count`, `provider_is_enriched`, and
  `enrichment_scope` are added as supporting indicators for later analysis and for
  tracking how much of the dataset now has a confirmed Wikidata-backed provider ID.


In [13]:
df_enriched["has_subject"] = df_enriched["subject"].notna()
df_enriched["has_country"] = df_enriched["country"].notna()

df_enriched["description_word_count"] = df_enriched["description"].fillna("").apply(
    lambda x: len(str(x).split()) if str(x).strip() != "" else 0
)

df_enriched["provider_is_enriched"] = df_enriched["match_status"] == "matched"

scope_map = {
    "matched": "provider_confirmed",
    "uncertain": "provider_candidate_unconfirmed",
    "needs_manual_review": "provider_unidentified",
    "not_checked": "provider_not_reviewed",
}
df_enriched["enrichment_scope"] = df_enriched["match_status"].map(scope_map)

print("has_subject counts:")
print(df_enriched["has_subject"].value_counts(dropna=False))
print()
print("has_country counts:")
print(df_enriched["has_country"].value_counts(dropna=False))
print()
print("description_word_count summary:")
print(df_enriched["description_word_count"].describe())
print()
print("enrichment_scope counts:")
print(df_enriched["enrichment_scope"].value_counts(dropna=False))


has_subject counts:
has_subject
False    245
Name: count, dtype: int64

has_country counts:
has_country
True    245
Name: count, dtype: int64

description_word_count summary:
count    245.000000
mean      16.840816
std       31.743903
min        0.000000
25%        0.000000
50%        2.000000
75%       19.000000
max      272.000000
Name: description_word_count, dtype: float64

enrichment_scope counts:
enrichment_scope
provider_confirmed                186
provider_not_reviewed              54
provider_unidentified               3
provider_candidate_unconfirmed      2
Name: count, dtype: int64


## Part 4 - Clickable Wikidata links for verified QIDs

Every value in `wikidata_qid` at this point was individually verified against
Wikidata.


In [14]:
import re

qid_pattern = re.compile(r"^Q\d+$")

def make_wikidata_url(qid):
    if pd.isna(qid):
        return pd.NA
    qid_str = str(qid).strip()
    if not qid_pattern.match(qid_str):
        return "INVALID_QID_FORMAT"
    return f"https://www.wikidata.org/wiki/{qid_str}"

df_enriched["wikidata_url"] = df_enriched["wikidata_qid"].apply(make_wikidata_url)

n_with_qid = df_enriched["wikidata_qid"].notna().sum()
n_with_url = df_enriched["wikidata_url"].notna().sum()
n_invalid = (df_enriched["wikidata_url"] == "INVALID_QID_FORMAT").sum()

print("Rows with a wikidata_qid:", n_with_qid)
print("Rows with a generated wikidata_url:", n_with_url)
print("Rows with an invalid/malformed QID (flagged, not linked):", n_invalid)
print()
print("Sample links:")
print(df_enriched.loc[df_enriched["wikidata_url"].notna(), ["dataProvider", "wikidata_qid", "wikidata_url"]].drop_duplicates(subset="wikidata_qid").head(10).to_string(index=False))

assert n_with_qid == n_with_url, "Every row with a QID must get a URL (or an explicit INVALID flag)."
assert n_invalid == 0, "No malformed QIDs should exist at this point - investigate if this fails."


Rows with a wikidata_qid: 188
Rows with a generated wikidata_url: 188
Rows with an invalid/malformed QID (flagged, not linked): 0

Sample links:
                             dataProvider wikidata_qid                            wikidata_url
                  Royal Museums Greenwich     Q7374509  https://www.wikidata.org/wiki/Q7374509
    National Audiovisual Institute France     Q1665109  https://www.wikidata.org/wiki/Q1665109
                Marciana National Library      Q578460   https://www.wikidata.org/wiki/Q578460
             University Library of Genova     Q3639620  https://www.wikidata.org/wiki/Q3639620
DFF – German Film Institute & Film Museum     Q1205634  https://www.wikidata.org/wiki/Q1205634
                         Cinecittà - Luce      Q781183   https://www.wikidata.org/wiki/Q781183
                      Cineteca di Bologna     Q1092493  https://www.wikidata.org/wiki/Q1092493
 Bodleian Libraries, University of Oxford    Q16147979 https://www.wikidata.org/wiki/Q16147979


## Part 5 - Save final consolidated dataset and validate

The original input file (`europeana_india_unique_titles_enhanced_top15_providers.csv`)
is left untouched. The result of this notebook is saved as a new file, following the
project's naming convention.


In [15]:
output_file = processed_dir / "europeana_india_unique_titles_enhanced_extended.csv"

df_enriched.to_csv(output_file, index=False, encoding="utf-8")

print("Input rows:", len(df))
print("Output rows:", len(df_enriched))
print("Input columns:", len(df.columns))
print("Output columns:", len(df_enriched.columns))
print("New columns added:", [c for c in df_enriched.columns if c not in df.columns])
print()

print("Match-status comparison (before this notebook -> after):")
before = df["match_status"].value_counts().reindex(
    ["matched", "uncertain", "not_checked"], fill_value=0)
after = df_enriched["match_status"].value_counts().reindex(
    ["matched", "uncertain", "not_checked", "needs_manual_review"], fill_value=0)
comparison = pd.DataFrame({"before": before, "after": after}).fillna(0).astype(int)
print(comparison)
print()

print("Output file:", output_file)
print("File exists:", output_file.exists())

assert len(df) == len(df_enriched), "Row count changed unexpectedly."
assert set(df.columns).issubset(set(df_enriched.columns)), "Original columns must all be preserved."
print()
print("All validation checks passed.")


Input rows: 245
Output rows: 245
Input columns: 16
Output columns: 25
New columns added: ['provider_enrichment_notes', 'record_relevance_status', 'relevance_notes', 'has_subject', 'has_country', 'description_word_count', 'provider_is_enriched', 'enrichment_scope', 'wikidata_url']

Match-status comparison (before this notebook -> after):
                     before  after
match_status                      
matched                 139    186
needs_manual_review       0      3
not_checked              78     54
uncertain                28      2

Output file: C:\Users\kevin\Documents\GitHub\Open_Project_2026\data\processed\europeana_india_unique_titles_enhanced_extended.csv
File exists: True

All validation checks passed.


## Limitations and methodological reflection

**On the `possible_false_positive` finding.** 17 of 245 records (~6.9%) captured by
this project's India-related search strategy turned out, on manual review, to be
about Indigenous peoples of the Americas rather than India. This is not incidental
noise - it traces to a specific, identifiable source: the German cataloguing term
"Indianer," used across several German institutions in this sample, collides with the
English search term "Indian" used in this project's query strategy. All 17 records
are retained in the dataset (not deleted) and are marked
`record_relevance_status = "likely_not_india_related"` so that later analysis can
exclude them from India-specific figures while keeping them visible for audit. The
228 remaining records were **not** individually re-verified as India-related; they
simply were not flagged as `possible_false_positive` by the earlier rule-based check,
and should not be read as individually confirmed.

**On the empty `subject` field.** `has_subject` is `False` for all 245 records in
this dataset. Europeana's data model (EDM) requires at least one of several
descriptive properties (subject, coverage, type, spatial, or temporal) to be present,
and `type` - a broad, cheap-to-assign platform category (IMAGE/TEXT/VIDEO/SOUND) - is
evidently what providers in this sample use to satisfy that requirement, rather than
`subject`, which requires real cataloguing labour and typically draws on a controlled
vocabulary such as the Getty Art & Architecture Thesaurus (AAT), the German Integrated
Authority File (GND), or the Library of Congress Subject Headings (LCSH).

These two findings are mechanistically connected: a populated `subject` field using a
controlled vocabulary would very likely have disambiguated "India" from "Indigenous
peoples of the Americas" at the cataloguing stage, since such vocabularies typically
maintain separate authority terms for exactly this distinction. Because `subject` is
empty across the board, discovery in this dataset depends entirely on free-text
matching against titles and descriptions - precisely where this kind of language
ambiguity lives. In other words, the false-positive rate observed here is not a flaw
in this project's search strategy so much as a demonstrable consequence of a gap
between what Europeana's schema is designed to support and what contributing
institutions in this sample actually deliver.

**On provider enrichment.** This notebook increased confirmed (`matched`) provider
identifications from 139 to 186 of 245 records, using only verifiable Wikidata
evidence with a documented reasoning note per provider. Two decisions were made
against the "maximise matches" instinct on purpose: one provider (National
Audiovisual Institute France) was *downgraded* from an initial confident match back
to `uncertain` once its exact provider string could not be verified verbatim, and one
provider (Map Collection UK) was left `needs_manual_review` rather than guessed. Both
choices reflect this project's core principle: preserve uncertainty rather than
overstate the reliability of metadata.
